In [ ]:
pip install torch torchvision torchaudio

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# --- Bước 1: Đọc dữ liệu và tiền xử lý ---
# Đọc dữ liệu từ file CSV, parse cột 'date' và sắp xếp theo thời gian
data = pd.read_csv('data.csv', parse_dates=['date'], index_col='date')
data = data.sort_index()

# Chỉ lấy cột 'value'
values = data[['value']].values

# Chuẩn hóa dữ liệu về khoảng [0,1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_values = scaler.fit_transform(values)

# Hàm tạo dữ liệu dạng sequence với time_steps cố định (ví dụ: 3 bước)
def create_sequences(dataset, time_steps=3):
    X, y = [], []
    for i in range(len(dataset) - time_steps):
        X.append(dataset[i:(i+time_steps), 0])
        y.append(dataset[i+time_steps, 0])
    return np.array(X), np.array(y)

time_steps = 3  # Sử dụng 3 bước thời gian trước để dự báo giá trị tiếp theo
X, y = create_sequences(scaled_values, time_steps)

# Chuyển đổi X và y sang định dạng phù hợp với PyTorch
X = X.reshape(X.shape[0], time_steps, 1)
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra (80%-20%)
split_idx = int(len(X_tensor) * 0.8)
train_X, test_X = X_tensor[:split_idx], X_tensor[split_idx:]
train_y, test_y = y_tensor[:split_idx], y_tensor[split_idx:]

train_dataset = TensorDataset(train_X, train_y)
test_dataset = TensorDataset(test_X, test_y)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# --- Bước 2: Xây dựng mô hình LSTM với PyTorch ---
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x có dạng [batch_size, seq_length, input_size]
        lstm_out, _ = self.lstm(x)
        # Lấy output tại timestep cuối cùng
        out = self.fc(lstm_out[:, -1, :])
        return out

input_size = 1    # Chỉ có 1 đặc trưng (value)
hidden_size = 50  # Số đơn vị ẩn
num_layers = 2    # Số lớp LSTM
output_size = 1   # Dự báo giá trị tiếp theo

model = LSTMModel(input_size, hidden_size, num_layers, output_size)
print(model)

# --- Bước 3: Định nghĩa hàm loss và optimizer ---
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- Bước 4: Huấn luyện mô hình ---
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.4f}")

# --- Bước 5: Dự báo và trực quan hóa kết quả ---
model.eval()
with torch.no_grad():
    # Dự báo trên tập kiểm tra
    predictions = model(test_X).detach().numpy()

# Chuyển đổi dữ liệu về giá trị ban đầu
predictions_inv = scaler.inverse_transform(predictions)
test_y_inv = scaler.inverse_transform(test_y.numpy())

plt.figure(figsize=(8, 5))
plt.plot(test_y_inv, label='Giá trị thực', marker='o')
plt.plot(predictions_inv, label='Dự báo', marker='x', linestyle='--')
plt.title('Dự báo chuỗi thời gian với LSTM (PyTorch)')
plt.xlabel('Mẫu')
plt.ylabel('Giá trị')
plt.legend()
plt.show()
